# 0B · Maths du lycée au ML — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch00b_maths/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 0B.1 à 0B.27 | vérifier tes exercices papier ✏️ | ✏️ | ★ à ★★ | |
| A · Nombres et fonctions en code | 0B.33 à 0B.37 | *prochaine session de génération* | 📦 🔮 🐛 | | |
| B · mylearn.linalg_basics : vecteurs et matrices en Python pur | 0B.38 à 0B.46 | *prochaine session de génération* | 🔨 📦 🔬 🔮 🐛 | | |
| C · Dérivées et gradients en code | 0B.47 à 0B.51 | *prochaine session de génération* | 🔨 📈 🔮 | | |
| D · Probabilités simulées et défi final | 0B.52 à 0B.54 | *prochaine session de génération* | 🔬 🔨 🏆 | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        if pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Vérifier tes calculs à la main (partie 0), puis les retrouver en code.
- Écrire l'algèbre linéaire en Python pur (`mylearn.linalg_basics`) et la comparer à NumPy.
- Estimer des dérivées et des gradients par des pentes numériques, et lire des lignes de niveau.
- Confirmer des probabilités, des espérances et des variances par simulation.

**Rappel express.** $\sum$ est une boucle ; $(m, n) \times (n, p) \to (m, p)$ ; $\frac{dz}{dx} = \frac{dz}{dy}\frac{dy}{dx}$ ; on descend en soustrayant le gradient ; en code, `np.log` est $\ln$ et `A @ B` est le produit matriciel (`A * B` est élément par élément).

## Partie 0 · Vérifier tes exercices papier (✏️ de 0B.1 à 0B.27)

Fais d'abord les exercices ✏️ de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée (par exemple `42`, `0.125`, `[7, -2]`, `[[1, 0], [0, 1]]`, `(4, 5)`, `True`), pas l'expression Python, sinon tu ne vérifies rien. Arrondis comme l'énoncé le demande. Les réponses pas encore remplies affichent ⏳.

In [2]:
# Data of the paper exercises (the same as in 02_exercices.md)
import math
from itertools import product

import numpy as np

x3 = [2, -1, 4, 3]                                                   # 0B.3
s4 = [2, 4, 9, 1, 5, 6]                                              # 0B.4
omega5 = set(range(1, 13))                                           # 0B.5
A5 = {n for n in omega5 if n % 2 == 0}
B5 = {n for n in omega5 if n % 3 == 0}
C5 = {1, 2, 3}
p1, p2, u8 = np.array([40, 190]), np.array([48, 196]), np.array([3, -4])   # 0B.8
A9 = np.array([[2, 0, 1], [-1, 3, 2]])                               # 0B.9
v9 = np.array([1, 2, -1])
X9 = np.array([[1, 2], [3, 0], [0, -1]])
w9 = np.array([0.5, 2])


def f10(x):                                                          # 0B.10
    return x ** 2 + x


DICE = list(product(range(1, 7), repeat=2))                         # 0B.11, 0B.26


def prob(event):
    """Probability of an event (a function of the pair of dice) with two fair dice."""
    return sum(1 for d in DICE if event(d)) / len(DICE)


def first_even(d):
    return d[0] % 2 == 0


def sum_is(total):
    return lambda d: d[0] + d[1] == total


def sigmoid(x):                                                      # 0B.17
    return 1 / (1 + math.exp(-x))


va, vb, vc, vd = (np.array(v) for v in ([1, 2, 2], [2, 0, -1], [2, 4, 4], [1, 0, 0]))   # 0B.18


def cosine(a, b):
    return (a @ b) / (np.linalg.norm(a) * np.linalg.norm(b))


A20 = np.array([[1, 2], [0, -1], [3, 1]])                            # 0B.20
B20 = np.array([[2, 1, 0], [1, -1, 4]])
C20 = np.array([[1, 1], [2, 0]])
M21 = np.array([[3.0, 1.0], [4.0, 2.0]])                             # 0B.21


def f25(x, y):                                                       # 0B.25
    return (x - 1) ** 2 + 2 * y ** 2


values27, probs27 = np.array([0, 1, 2, 5]), np.array([0.4, 0.3, 0.2, 0.1])   # 0B.27

**Ex 0B.1 — Puissances, racines et notation scientifique sans calculatrice**

In [3]:
wb.record("0B.1a", 2 ** 5 * 2 ** 3, mistakes={"on ADDITIONNE les exposants : 2^(5+3)": 2 ** 15})
wb.record("0B.1b", (10 ** 2) ** 3 / 10 ** 4, decimals=0, mistakes={"(10^2)^3 = 10^6 : on MULTIPLIE les exposants": 10})
wb.record("0B.1c", 5 ** -2, decimals=2, mistakes={"un exposant négatif donne un inverse, pas un nombre négatif": -25})
wb.record("0B.1d", math.sqrt(49 * 16), decimals=0)
wb.record("0B.1e", round(8 ** (2 / 3)), mistakes={"8^(1/3) = 2 (racine cubique), puis 2^2": 2})
wb.record("0B.1f", 3.2e5 * 2e-3, decimals=0)
wb.record("0B.1g", math.floor(math.log10(0.00056)), mistakes={"pour un nombre plus petit que 1, l'exposant est négatif": 4})
wb.record("0B.1h", round(30 * math.log10(2)), mistakes={"on cherche l'exposant de 10, pas celui de 2 : 2^30 = (2^10)^3 et 2^10 ≈ 10^3": 30})

WB_ANSWER {"hash": "0d51f482a1d5a7103acd9e4cc820fdd77581c864d6419aa16beec4df8b3e62af", "id": "0B.1a", "kind": "int", "magnitude": 2, "mistakes": {"f5ba6170bce85fea52e2c8e53fc634efb5d222b1c67fad029c1e8a0decd1c5ff": "on ADDITIONNE les exposants : 2^(5+3)"}, "sign": 1}
📝 Ex 0B.1a : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "hash": "5da56270075d9f428261749bbd4c46d4f5cb82cfecae7a15bca6745155e0870a", "id": "0B.1b", "kind": "float", "magnitude": 2, "mistakes": {"a793cee87d221a442fe3cbf5bdc4940ce8a4b93c501584d44594460e69f93d03": "(10^2)^3 = 10^6 : on MULTIPLIE les exposants"}, "sign": 1}
📝 Ex 0B.1b : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "1560abe06013657987b528d83b6fd798824a7d69aedce4a27c8d28b934766523", "hash_coarse": "180f69baafd380dd89c0dbc86a44d0a90214f8d579b92ec21bc8791b0417baeb", "id": "0B.1c", "kind": "float", "magnitude": -2, "mistakes": {"d46ee241d38924195d39c03906dc4c5fe8acd6e47e00550a053374e139d12e30": "un exposant négatif donne un 

**Ex 0B.2 — Valeur absolue, partie entière et signe : tableau de valeurs**

In [4]:
wb.record("0B.2a", abs(-7) + abs(3 - 5), mistakes={"une valeur absolue est toujours positive : |3 − 5| = 2": 5})
wb.record("0B.2b", math.floor(3.7))
wb.record("0B.2c", math.floor(-3.7), mistakes={"le plancher va vers le BAS (vers −∞) : prends l'entier juste en dessous de −3,7": -3})
wb.record("0B.2d", math.ceil(-3.7), mistakes={"le plafond va vers le HAUT : prends l'entier juste au-dessus de −3,7": -4})
wb.record("0B.2e", math.ceil(2.01))
wb.record("0B.2f", int(np.sign(-0.5) + np.sign(0) + np.sign(12)), mistakes={"sign(0) vaut 0": 1})
wb.record("0B.2g", 4 - 1.5, decimals=1)
wb.record("0B.2h", 4 + 1.5, decimals=1)
wb.record("0B.2i", sum(1 for x in range(-10, 11) if abs(x) < 3), mistakes={"inégalité stricte : 3 et −3 sont exclus": 7, "les entiers négatifs comptent aussi": 3})

WB_ANSWER {"hash": "7a61f56c4ac9fd30230d07c53b7c4a0c33db86570df56b379ada4c250c993075", "id": "0B.2a", "kind": "int", "magnitude": 0, "mistakes": {"7d542dfb3b7258987a6c2d11ced875501f7bf1d9cd3ee05c0fff1d3b61a2c751": "une valeur absolue est toujours positive : |3 − 5| = 2"}, "sign": 1}
📝 Ex 0B.2a : réponse enregistrée (int).
WB_ANSWER {"hash": "40f8108fa1f6e3f6dc1549005533c81e2d67ef053ef14795d2069024c069a9af", "id": "0B.2b", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 0B.2b : réponse enregistrée (int).
WB_ANSWER {"hash": "5d6ae8b95d0ef081078e8c3321a2656e761843688d063f65d720b5fc36b3881a", "id": "0B.2c", "kind": "int", "magnitude": 0, "mistakes": {"4937808cfb3fed9e204b51cd7e320bad62e47369c8ca5e4498540b0f764545a4": "le plancher va vers le BAS (vers −∞) : prends l'entier juste en dessous de −3,7"}, "sign": -1}
📝 Ex 0B.2c : réponse enregistrée (int).
WB_ANSWER {"hash": "30728b838032d176bd1d934a6cfc94c0408aebfc125fc111e9d8ab86465b823f", "id": "0B.2d", "kind": "int", "magnitude": 0, "mistakes

**Ex 0B.3 — Lire et calculer des Σ et des Π**

In [5]:
wb.record("0B.3a", sum(range(1, 6)))
wb.record("0B.3b", sum(2 * i - 1 for i in range(1, 5)))
wb.record("0B.3c", sum(3 ** k for k in range(4)), mistakes={"k commence à 0 : le terme 3^0 = 1 compte aussi": 39})
wb.record("0B.3d", math.prod(i + 1 for i in range(1, 5)))
wb.record("0B.3e", sum(v ** 2 for v in x3), mistakes={"le carré de −1 vaut +1": 28})
wb.record("0B.3f", sum(3 * v + 1 for v in x3), mistakes={"le +1 est ajouté à chacun des 4 termes, pas une seule fois": 25})
wb.record("0B.3g", 5 * 100)
wb.record("0B.3h", sum(i * x3[i - 1] for i in range(2, 5)), mistakes={"la somme commence à i = 2 : le terme 1 × x_1 n'y est pas": 24})

WB_ANSWER {"hash": "9b434400e82cf32bd26f6ff3bd47cca8cb65a99a10088c916696720e4c10d627", "id": "0B.3a", "kind": "int", "magnitude": 1, "sign": 1}
📝 Ex 0B.3a : réponse enregistrée (int).
WB_ANSWER {"hash": "ebbc7b814ebfc4db0b864b0585d32bf195417aa719d12c78339365c08470196d", "id": "0B.3b", "kind": "int", "magnitude": 1, "sign": 1}
📝 Ex 0B.3b : réponse enregistrée (int).
WB_ANSWER {"hash": "b78004d6d0eff20a0d1ffa2c4f55bc036c0c6309e986167a5df3f483d5fdc46a", "id": "0B.3c", "kind": "int", "magnitude": 1, "mistakes": {"e26efcab5b5e9e47f7b1b9d34cedd6800451bb841487d9c24e5a2230dc14f990": "k commence à 0 : le terme 3^0 = 1 compte aussi"}, "sign": 1}
📝 Ex 0B.3c : réponse enregistrée (int).
WB_ANSWER {"hash": "7fc5913a13319c28f72714520e18b35ca0af929ddd05c556fc195c3afb530285", "id": "0B.3d", "kind": "int", "magnitude": 2, "sign": 1}
📝 Ex 0B.3d : réponse enregistrée (int).
WB_ANSWER {"hash": "f749103a27f1ac3ffe68acfd8af074b7e0758882267479121ebb96d2337f3e3c", "id": "0B.3e", "kind": "int", "magnitude": 1,

**Ex 0B.4 — Moyenne pondérée, somme pondérée et moyenne mobile à la main**

In [6]:
wb.record("0B.4a", np.mean([4, 8, 6, 10, 2]), decimals=0)
wb.record("0B.4b", np.average([14, 8, 11], weights=[3, 1, 2]), decimals=0, mistakes={"divise par la somme des coefficients (6), pas par le nombre de notes": 24})
wb.record("0B.4c", 0.2 * 10 - 0.5 * 4 + 1 * 3 - 1, decimals=1, mistakes={"n'oublie pas le biais b = −1": 3})
wb.record("0B.4d", [round(np.mean(s4[t - 3:t]), 2) for t in range(3, 7)], decimals=2)
wb.record("0B.4e", 50 - 7 + 1, mistakes={"la première moyenne mobile existe à t = k : il y a n − k + 1 valeurs": 43})

WB_ANSWER {"decimals": 0, "hash": "828a5bed7d1429bd7f1edd596fca0f85cfb197a12bfb700a8b357715d861f482", "id": "0B.4a", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.4a : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "hash": "cfbd96e25aac9967155a4f74553420ce0041952ef4062ae722e00efcb7adc661", "id": "0B.4b", "kind": "float", "magnitude": 1, "mistakes": {"177287f1d313cdfb4f7ef0f4144b69364646849dcb9f297311de60583769be01": "divise par la somme des coefficients (6), pas par le nombre de notes"}, "sign": 1}
📝 Ex 0B.4b : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "e92823d2ffb79457eb255fe66e7cdff8f9940757ee614bc8bc653ce6b3a4acad", "hash_coarse": "171052ea36c3485b17bf912322758967bb698e00658cd1286612bd34cbaba626", "id": "0B.4c", "kind": "float", "magnitude": 0, "mistakes": {"f0d493d3b55c4b7358efc19e76557f35e028da7c50e0504e988333cb281c60c7": "n'oublie pas le biais b = −1"}, "sign": 1}
📝 Ex 0B.4c : réponse enregistrée (float, 1 décimale(

**Ex 0B.5 — Ensembles : union, intersection, complémentaire et cardinal**

In [7]:
wb.record("0B.5a", len(A5 & B5))
wb.record("0B.5b", len(A5 | B5), mistakes={"retire l'intersection, comptée deux fois": 10})
wb.record("0B.5c", len(omega5 - A5))
wb.record("0B.5d", len((omega5 - A5) & (omega5 - B5)), mistakes={"ni multiple de 2 NI multiple de 3 : c'est le complémentaire de A ∪ B": 8, "« ni… ni… » exclut aussi les nombres qui ne sont que dans A, ou que dans B": 10})
wb.record("0B.5e", A5 & C5, ordered=False)
wb.record("0B.5f", len(A5 | B5 | C5), mistakes={"2 et 3 sont déjà dans A ∪ B : seul 1 est nouveau": 11})

WB_ANSWER {"hash": "1281fee28000528462fc0f6ca3c31aa2c319d0512ccc28565dbc92750123c641", "id": "0B.5a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 0B.5a : réponse enregistrée (int).
WB_ANSWER {"hash": "bfe46db0c4d40f5b8d1fd8350d4c82a3d4eb48331d5430465e29f5c6f6ab23bf", "id": "0B.5b", "kind": "int", "magnitude": 0, "mistakes": {"8a85f989066221c550977e78c42df41b94af10eea5477a17fadedb8073b10a8f": "retire l'intersection, comptée deux fois"}, "sign": 1}
📝 Ex 0B.5b : réponse enregistrée (int).
WB_ANSWER {"hash": "4c076e622fc2c36d84b5a0af3796328a2a328e87eea5536e1889a247767b165c", "id": "0B.5c", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 0B.5c : réponse enregistrée (int).
WB_ANSWER {"hash": "fb5b4835f33cd71827a6182bb7d77388a7de604dfc0bb743a2633f4a4f13fcf5", "id": "0B.5d", "kind": "int", "magnitude": 0, "mistakes": {"5ced42762137a8f446d69b8e4e0553f1a60c5b0c57a50f40d854bb2190b255cd": "« ni… ni… » exclut aussi les nombres qui ne sont que dans A, ou que dans B", "f81c751750fc4cb6d8558bf6c2d51

**Ex 0B.6 — Droites et paraboles : pente, ordonnée à l'origine, racines, sommet**

In [8]:
wb.record("0B.6a", (-3 - 5) / (3 - (-1)), decimals=0, mistakes={"pente = Δy / Δx, avec les points pris dans le même ordre en haut et en bas": 2})
wb.record("0B.6b", 5 - (-2) * (-1))
wb.record("0B.6c", -3 / -2, decimals=1)
wb.record("0B.6d", (-8) ** 2 - 4 * 2 * 6, mistakes={"Δ = b² − 4ac avec b = −8 : (−8)² = 64": -112})
wb.record("0B.6e", (8 - math.sqrt(16)) / (2 * 2), decimals=0)
wb.record("0B.6f", (8 + math.sqrt(16)) / (2 * 2), decimals=0)
wb.record("0B.6g", 8 / (2 * 2), decimals=0, mistakes={"le sommet est en −b/(2a), avec b = −8": -2})
wb.record("0B.6h", 2 * 2 ** 2 - 8 * 2 + 6)

WB_ANSWER {"decimals": 0, "hash": "0c29d5e84e49baa105cf0f2b41c60961c9cc1a06076877d9aa960d9f8d88c99d", "id": "0B.6a", "kind": "float", "magnitude": 0, "mistakes": {"8fbaf5652d7b1b41e15810c9a6f749fe91506edf009ce32eeb23152d977e1581": "pente = Δy / Δx, avec les points pris dans le même ordre en haut et en bas"}, "sign": -1}
📝 Ex 0B.6a : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"hash": "1e8e74f4a35d0d9411df7672550503571038929269897d4d536b14d8d41cf559", "id": "0B.6b", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 0B.6b : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "dcd22ca9fd764e6fd03701a7a111b51fe18a4dc1e2e7ff8a15e423d3431630d6", "hash_coarse": "32bd160eaee5efafb36255318a43b85a5dc6acfc1f3c0ba184c6d8fc4314cd5d", "id": "0B.6c", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.6c : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"hash": "ce4004cc506f49f6301dab822e31ad314a812c095188228c2e766c9b80ecd664", "id": "0B.6d", "kind": "int", "magnitude": 1, "mi

**Ex 0B.7 — Cosinus : cercle, période et planning en cosinus**

In [9]:
wb.record("0B.7a", math.radians(60), decimals=2, mistakes={"π rad = 180°, donc 60° = π/3": 60 / 180})
wb.record("0B.7b", math.cos(math.pi), decimals=1)
wb.record("0B.7c", math.cos(2 * math.pi / 3), decimals=1)
wb.record("0B.7d", math.cos(-math.pi / 3), decimals=1, mistakes={"le cosinus est pair : cos(−x) = cos(x)": -0.5})
wb.record("0B.7e", math.cos(4 * math.pi), decimals=1, mistakes={"cos a une période de 2π : 4π, c'est deux tours complets": -1})
wb.record("0B.7f", 0.5 * (1 + math.cos(math.pi * 50 / 100)), decimals=2)
wb.record("0B.7g", 0.5 * (1 + math.cos(math.pi * 25 / 100)), decimals=2)
wb.record("0B.7h", 0.01 * 0.5 * (1 + math.cos(math.pi * 75 / 100)), decimals=4, mistakes={"multiplie le facteur par le learning rate maximal 0,01": 0.5 * (1 + math.cos(math.pi * 75 / 100)), "multiplie le facteur (≈ 0,146) par le learning rate maximal 0,01": 0.146, "multiplie le facteur (≈ 0,1465) par le learning rate maximal 0,01": 0.1465, "cos(3π/4) est négatif : cos(π − x) = −cos x": 0.0085})

WB_ANSWER {"decimals": 2, "hash": "46cdc23915db88ac01d9c076aaa7d0eaecd7aedbd324a1d18e65e491c3065ef8", "hash_coarse": "fd3bbd78d26f35c6a995c53c26b796b1bf685db9169f68e6af07dac4b35e86ff", "id": "0B.7a", "kind": "float", "magnitude": 0, "mistakes": {"9e0694877de82cf8ea669949afc14dd659ebe13bbce62fc41783faf20acd99bf": "π rad = 180°, donc 60° = π/3"}, "sign": 1}
📝 Ex 0B.7a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "c9e0ebc7d6dfdf5619ea00987c519e7777887a22888910c0a359f0613824476a", "hash_coarse": "86725fb7c27ff1085fbc1c29f7d80478b1e41ad782de9a877ccc0c729ea3e26e", "id": "0B.7b", "kind": "float", "magnitude": 0, "sign": -1}
📝 Ex 0B.7b : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "0e5d0a62d0dfd84f5972fa5f4577b2b4237423c3161f3e0e433057e2c3cde161", "hash_coarse": "6d0e5c480914cd10e8092ad29c94de11585184be074ab3d0f65c4ab6a5bc5551", "id": "0B.7c", "kind": "float", "magnitude": -1, "sign": -1}
📝 Ex 0B.7c : réponse enregistrée (flo

**Ex 0B.8 — Vecteurs : somme, multiple, norme et distance entre deux manchots**

In [10]:
wb.record("0B.8a", p1 + p2)
wb.record("0B.8b", (p1 + p2) / 2, decimals=1)
wb.record("0B.8c", np.linalg.norm(u8), decimals=1, mistakes={"7 est la norme L1 (somme des valeurs absolues) ; L2 : racine de la somme des carrés": 7})
wb.record("0B.8d", int(np.abs(u8).sum()), mistakes={"L1 : somme des VALEURS ABSOLUES": -1})
wb.record("0B.8e", int(np.abs(u8).max()), mistakes={"L∞ : la plus grande valeur ABSOLUE": 3})
wb.record("0B.8f", np.linalg.norm(p1 - p2), decimals=1)
wb.record("0B.8g", np.linalg.norm(np.array([40, 19]) - np.array([48, 19.6])), decimals=2)
wb.record("0B.8h", u8 / np.linalg.norm(u8), decimals=1)

WB_ANSWER {"decimals": 0, "element_hashes": ["49d5db091da5999a3dddb4ce580ab3d1342a46a56fad43ba0409fd774c869539", "473c2141562a6e0bbc801e17f659300685fd52bed20e45dc0f8201a13e28e27b"], "hash": "09476ed4d27407efbf3fae444becd23b70c5522fbf426b46756088a1c2d84b35", "id": "0B.8a", "integer": true, "kind": "array", "shape": [2]}
📝 Ex 0B.8a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 1, "element_hashes": ["e6d22b51b6e6e774774a33a0faca935c7138ce401d6589989177b6511b5ef16c", "269530142ca6115e55c28be803001bf64c2f733ad22faf8af52f57dab5cfef25"], "hash": "3316f48b17b6f7e8e706b83bef41bb15b763444503dead3755fd7dad30382b84", "id": "0B.8b", "kind": "array", "shape": [2]}
📝 Ex 0B.8b : réponse enregistrée (array, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "579897efdf6ac64ba44bac6440971cfe36279f754115f6a2d9735c72d81f8b48", "hash_coarse": "300d6348b5a2bb88a6170756bd7697562aeb551b54b0e2498059096ae12ba4d2", "id": "0B.8c", "kind": "float", "magnitude": 0, "mistakes": {"55e01e15b4c2e4

**Ex 0B.9 — Transposée et produit matrice-vecteur : deux lectures**

In [11]:
wb.record("0B.9a", A9.T.shape, mistakes={"la transposée échange lignes et colonnes": (2, 3)})
wb.record("0B.9b", A9.T[2, 0])
wb.record("0B.9c", A9 @ v9)
wb.record("0B.9d", A9.T @ np.array([1, -1]))
wb.record("0B.9e", A9.shape[1] == 2)
wb.record("0B.9f", X9 @ w9 + 1, decimals=1, mistakes={"n'oublie pas le biais b = 1": X9 @ w9})

WB_ANSWER {"decimals": 0, "element_hashes": ["0544fd55b0b22733497f8d32670e55aa8a519c09dcf720b55fce6ea7bf888ebf", "58f372537a95a1481f01baef1c49590a7fb3c787a1c51544ea4c968f0453c955"], "hash": "08630a4e13daee688be6e4ce2b7d4e1a1fa1bf523df396d7fb7eda03ec7351f3", "id": "0B.9a", "integer": true, "kind": "array", "mistakes": {"947631601bd3b9ff0f807881d1ffab25e7820a5b23b4acb24eea31d6502f8ba7": "la transposée échange lignes et colonnes"}, "shape": [2]}
📝 Ex 0B.9a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"hash": "5433c199277dfab2a94529e56d5edae74504f454db4e9acdadefdbd784665dbd", "id": "0B.9b", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 0B.9b : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "element_hashes": ["8f6caae443e0c7fd5946db865d2b8a62b0bce39e638bdb96bd923b7f3edb4ee0", "4b1ab4180bf2a5d358794cdcc7883aa6cfe58a9f4392fb510f9541cd374cfba4"], "hash": "9da1ddf7b61718033c8d7713e879a4b882cd39d1ba73d4eeeb1b57baf744fdb2", "id": "0B.9c", "integer": true, "kind": "array", "shap

**Ex 0B.10 — Taux d'accroissement : de la sécante à la tangente**

In [12]:
wb.record("0B.10a", (f10(3) - f10(2)) / 1, decimals=0)
wb.record("0B.10b", (f10(2.1) - f10(2)) / 0.1, decimals=1)
wb.record("0B.10c", (f10(2.01) - f10(2)) / 0.01, decimals=2)
wb.record("0B.10d", 2 * 2 + 1, mistakes={"développe f(2 + h) − f(2), divise par h, puis fais tendre h vers 0": 4})
wb.record("0B.10e", f10(2) - 5 * 2)
wb.record("0B.10f", f10(2) + 5 * 0.05, decimals=4, mistakes={"c'est la vraie valeur f(2,05) : on demande l'approximation donnée par la tangente": f10(2.05)})

WB_ANSWER {"decimals": 0, "hash": "fbb630922cdc0fb9467af1d8758de4b71cd1a5acc8cb04b16257c548c25ccf43", "id": "0B.10a", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.10a : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "8717b55a47c5c64f7e9e29ac47fe9e16ea2c13c30e2d9fa01cdcf1b057421f2f", "hash_coarse": "46253ac3f55eb1d8ee5fd50e4f3703190838a2c398a5c353e48858363b90c5d1", "id": "0B.10b", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.10b : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "84ec2a85885d895687499ab5a1f4968af1648122147d32ad1a25cbb66e164f35", "hash_coarse": "6dfa88a21b90230e0ea993a77a92bbe47d1db96c9981132e86c8191e6196f8ae", "id": "0B.10c", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.10c : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"hash": "2d2541605420ae3079fbb8baaa3cb3dc5772cb103977eb8e7b9fbba38f025e45", "id": "0B.10d", "kind": "int", "magnitude": 0, "mistakes": {"35724ca025cfd4f5d4a5d90d8e

**Ex 0B.11 — Probabilités : issues, complémentaire, union**

In [13]:
wb.record("0B.11a", prob(lambda d: sum(d) == 8), decimals=3)
wb.record("0B.11b", prob(lambda d: d[0] == d[1]), decimals=3)
wb.record("0B.11c", prob(lambda d: 1 in d), decimals=3, mistakes={"au moins un 1 = 1 − P(aucun 1) = 1 − (5/6)²": 2 / 6})
wb.record("0B.11d", prob(lambda d: sum(d) == 8 or d[0] == d[1]), decimals=3, mistakes={"(4, 4) est à la fois un double et une somme 8 : ne le compte qu'une fois": 11 / 36})
wb.record("0B.11e", prob(lambda d: sum(d) >= 10), decimals=3)
wb.record("0B.11f", (13 + 12 - 3) / 52, decimals=3, mistakes={"les 3 figures de cœur sont comptées deux fois : retire-les une fois": 25 / 52})

WB_ANSWER {"decimals": 3, "hash": "bf6817d7e02d76255dea39d64699c6109d04bb7b223a01e683fe2c47da5bcc2b", "hash_coarse": "f2526f48d36da4610cba2ae6e7eb7883865cfba25e846201285aad0688917d77", "id": "0B.11a", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.11a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "87a6da29ec212a7861d666a4844d572f223a481f830ec478277659435aec4ada", "hash_coarse": "ef6609f97095592c3005b806ad154b0a30b67110905cc18a2e994e7f8fe648de", "id": "0B.11b", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.11b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "7cb35d810da976870c435533c616ebe389ac59472d86e6fb13f2c632bc49ae40", "hash_coarse": "5e13c6bea4371337c9ceb634067cd6c34151902ccf390a6d850a21d950fe197a", "id": "0B.11c", "kind": "float", "magnitude": -1, "mistakes": {"8e222237b9f60be07f325660b0ed64070d4f86d7695b9959648665490e5118c2": "au moins un 1 = 1 − P(aucun 1) = 1 − (5/6)²"}, "sign": 1}
📝 Ex 0B.11c : rép

**Ex 0B.12 — Dénombrer : choix successifs, factorielle et C(n, k)**

In [14]:
wb.record("0B.12a", 10 ** 4)
wb.record("0B.12b", 10 * 9 * 8 * 7, mistakes={"tous différents : 10 choix, puis 9, puis 8, puis 7": 10 * 9 * 8})
wb.record("0B.12c", math.factorial(6))
wb.record("0B.12d", math.comb(6, 2), mistakes={"C(n, k) ne tient pas compte de l'ordre : divise par 2!": 30})
wb.record("0B.12e", math.comb(8, 3))
wb.record("0B.12f", math.comb(5, 2))
wb.record("0B.12g", 2 ** 5, mistakes={"n'oublie ni l'ensemble vide ni l'ensemble complet": 30})
wb.record("0B.12h", 10 * math.comb(9, 2), mistakes={"le président est un rôle à part : choisis-le, puis les deux autres membres": math.comb(10, 3)})

WB_ANSWER {"hash": "a944b9d5f4191644923d93fd68ce0e6db33187d53fa5acd830d2513c00d3e14e", "id": "0B.12a", "kind": "int", "magnitude": 4, "sign": 1}
📝 Ex 0B.12a : réponse enregistrée (int).
WB_ANSWER {"hash": "d06d5cc45c4fc38967d009ac02ce7cf188e7dafb8e8fcfa578d7d429fc12c046", "id": "0B.12b", "kind": "int", "magnitude": 3, "mistakes": {"c17e8eab4a7ddd6c5f79feea9ce3563cfd2fb456498cccddcd2fe284a01a1bdf": "tous différents : 10 choix, puis 9, puis 8, puis 7"}, "sign": 1}
📝 Ex 0B.12b : réponse enregistrée (int).
WB_ANSWER {"hash": "d81a1bdced9c088de18eb66351616ccc98c3eb375e5bc9a49145ccb581782f4f", "id": "0B.12c", "kind": "int", "magnitude": 2, "sign": 1}
📝 Ex 0B.12c : réponse enregistrée (int).
WB_ANSWER {"hash": "f82387d62863fca8b4d9df31f039a9bb7d839147f35bd6920c0778e692cd00c2", "id": "0B.12d", "kind": "int", "magnitude": 1, "mistakes": {"b2e9ec663dc403ebfab2a8828c742e91240f91a1a5e2d83811deb8cec0087f8a": "C(n, k) ne tient pas compte de l'ordre : divise par 2!"}, "sign": 1}
📝 Ex 0B.12d : réponse

**Ex 0B.13 — Suites géométriques : ce qui fond, ce qui explose**

In [15]:
wb.record("0B.13a", 1000 * 0.5 ** 5, decimals=2, mistakes={"u_5 = u_0 × q^5 (5 multiplications depuis u_0)": 1000 * 0.5 ** 4})
wb.record("0B.13b", 0.9 ** 20, decimals=3)
wb.record("0B.13c", 1.1 ** 10, decimals=2, mistakes={"1,1^10 n'est pas 1 + 10 × 0,1 : les intérêts se cumulent": 2.0})
wb.record("0B.13d", (1 - 0.5 ** 10) / (1 - 0.5), decimals=3, mistakes={"de k = 0 à 9, il y a 10 termes : q^n avec n = 10": (1 - 0.5 ** 9) / (1 - 0.5)})
wb.record("0B.13e", 1 / (1 - 0.95), decimals=0)
wb.record("0B.13f", next(k for k in range(100) if 0.5 ** k < 0.001))
wb.record("0B.13g", (-0.8) ** 7, decimals=3, mistakes={"une raison négative à une puissance IMPAIRE donne un nombre négatif": 0.8 ** 7})

WB_ANSWER {"decimals": 2, "hash": "9b35fe960ebae597472b1aa32aeb7f4e1e21b23065f0c4a5e743528a64693558", "hash_coarse": "ad5693bd946c31f4574a7efc3af03be3826fd16eef9925a3986badc6b7be83b6", "id": "0B.13a", "kind": "float", "magnitude": 1, "mistakes": {"b0225dcb0ddb45161fada9d5e7304030ee0cbec19521cfc7199c962ea343cbb4": "u_5 = u_0 × q^5 (5 multiplications depuis u_0)"}, "sign": 1}
📝 Ex 0B.13a : réponse enregistrée (float, 2 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "5814379b98c6052ce84c848c765b1b971b31b72a1ab380412d01d688a1537a47", "hash_coarse": "172805fe133b805e612eae75eb476171ec48eb7752a15ec6884185fdb6f56ded", "id": "0B.13b", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.13b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "5af2d92f543ca51ae74549a421531c3b1ba2a1d94931938ffd1ee675a006d8ce", "hash_coarse": "ced6f748caef266cb8e52670b4575fc01c8216d06f368805e635c1b98392e375", "id": "0B.13c", "kind": "float", "magnitude": 0, "mistakes": {"8013aa5200a05

**Ex 0B.15 — Exponentielles et logarithmes : règles de calcul en bases 2, e et 10**

In [16]:
wb.record("0B.15a", math.log(math.e ** 3), decimals=0)
wb.record("0B.15b", math.exp(math.log(5)), decimals=0)
wb.record("0B.15c", math.log(1) + math.log2(32), decimals=0, mistakes={"ln 1 = 0 (et non 1)": 6})
wb.record("0B.15d", math.log10(0.001), decimals=0)
wb.record("0B.15e", math.log(math.e ** 2 * math.e ** 5), decimals=0, mistakes={"e^2 × e^5 = e^(2+5) : on additionne les exposants": 10})
wb.record("0B.15f", math.log(20), decimals=2)
wb.record("0B.15g", math.log2(1000), decimals=2, mistakes={"x = log₂ 1000 = ln 1000 / ln 2, pas log₁₀ 1000": 3.0, "10 est le nombre de bits (arrondi à l'entier supérieur) ; ici on demande la solution x, à 2 décimales": 10})
wb.record("0B.15h", 100 * math.log(0.01), decimals=1)
wb.record("0B.15i", 1 / math.log(2), decimals=3, mistakes={"1 nat vaut 1 / ln 2 bit, et non ln 2": math.log(2)})

WB_ANSWER {"decimals": 0, "hash": "8a67de68ef7bba9fabacf4da21cac2acd8c1d267077993dcdb6ae19f23304121", "id": "0B.15a", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.15a : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "hash": "f278376b4983f840b836ec12f3863a19c44e59454aee3f65fd48e6a41f17399e", "id": "0B.15b", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.15b : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "hash": "459c834f4810df66bfc647ed54f16d967e0bc771d8b895a36749c294568e6b60", "id": "0B.15c", "kind": "float", "magnitude": 0, "mistakes": {"d8fba7370c6dac0c4bc15b3f4a6c8d9ef6465bb7febe2f04acfbfa5c64c5836b": "ln 1 = 0 (et non 1)"}, "sign": 1}
📝 Ex 0B.15c : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "hash": "9b14b23d28ca90f7272770d8ca87843ece2abdf6d74918fe63d475c4097e0f30", "id": "0B.15d", "kind": "float", "magnitude": 0, "sign": -1}
📝 Ex 0B.15d : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"de

**Ex 0B.17 — Sigmoïde et tanh : valeurs, limites, symétries**

In [17]:
wb.record("0B.17a", sigmoid(0), decimals=3)
wb.record("0B.17b", sigmoid(3), decimals=3)
wb.record("0B.17c", sigmoid(-3), decimals=3, mistakes={"σ(−x) = 1 − σ(x), et non −σ(x)": -sigmoid(3)})
wb.record("0B.17d", math.tanh(0.5), decimals=3)
wb.record("0B.17e", 2 * sigmoid(1) - 1, decimals=3)
wb.record("0B.17f", 0, mistakes={"quand x → −∞, e^(−x) explose et σ(x) → 0": 1})
wb.record("0B.17g", math.log(9), decimals=2, mistakes={"σ(x) = 0,9 ⟺ e^(−x) = 1/9 ⟺ x = ln 9": -math.log(9)})
wb.record("0B.17h", round(sigmoid(1.7) + sigmoid(-1.7)))

WB_ANSWER {"decimals": 3, "hash": "1dfb0ed975a3027f5a34ff1e035a4b53de2cbcbfea3ae62d3a6edeb8732edec7", "hash_coarse": "05db1da2704338b7c95d2e5f738be31c9d4fc936b7b40fb2e656b646f03b1da0", "id": "0B.17a", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.17a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "441d7bab2e88344df84d552a71fedf9c7f1fc7e179795805867a25458ad7a44b", "hash_coarse": "85c02dcb479bd5e1542fc4e494867340c2eb085281052ddc608ce7c68518b6ea", "id": "0B.17b", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.17b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "7be5fe83fc89e0036c0171edf45c93bbcf822c0581de84191ce1f95145f2b79b", "hash_coarse": "5a18fa407e8596ddded23f99ea25d5a03fa1118ec956dfa1b135406b7aff89e1", "id": "0B.17c", "kind": "float", "magnitude": -2, "mistakes": {"067f096d80038706aceafd02df99c5536d962a6df82a5dccebec35259c9b70f7": "σ(−x) = 1 − σ(x), et non −σ(x)"}, "sign": 1}
📝 Ex 0B.17c : réponse enregist

**Ex 0B.18 — Produit scalaire, similarité cosinus et produit de Hadamard**

In [18]:
wb.record("0B.18a", int(va @ vb))
wb.record("0B.18b", bool(va @ vb == 0))
wb.record("0B.18c", np.linalg.norm(va), decimals=1)
wb.record("0B.18d", int(va @ vc))
wb.record("0B.18e", cosine(va, vc), decimals=1)
wb.record("0B.18f", cosine(va, vd), decimals=3, mistakes={"divise par les DEUX normes : ‖a‖ = 3 et ‖d‖ = 1": 1.0})
wb.record("0B.18g", va * vb)
wb.record("0B.18h", int((va * vc).sum()))
wb.record("0B.18i", cosine(va, -va), decimals=1)

WB_ANSWER {"hash": "4c72b6590d761806d7f7be1692e0c09b7331c068217b5ea6a913e2db5a9c6287", "id": "0B.18a", "kind": "int", "magnitude": null, "sign": 0}
📝 Ex 0B.18a : réponse enregistrée (int).
WB_ANSWER {"hash": "d59f5b105d30191056c32bd7053fb8ffd3c2ed641aa29c655a91891914ab3682", "id": "0B.18b", "kind": "bool"}
📝 Ex 0B.18b : réponse enregistrée (bool).
WB_ANSWER {"decimals": 1, "hash": "98586f8fac4c951b5917191d4933461bf2e06f6b61a66c58901a0677d3b0c976", "hash_coarse": "fe38a7d68b8db5c194ef71dae761b9b9771ae756f057895ff063d81e3fa4e1b8", "id": "0B.18c", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.18c : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"hash": "ee8316a3c0ab72341102f92f5ec1da72849b4dabafad84e9ab7bd8725bb18100", "id": "0B.18d", "kind": "int", "magnitude": 1, "sign": 1}
📝 Ex 0B.18d : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "ce3dc3e5a2f83e5513fab00ffd4fb5e8ab7309cbe9cd347acd52e87d5e6cdf05", "hash_coarse": "851f7a391f7e3c1fe5cfd7346ef051aa7607dbbb733

**Ex 0B.20 — Produit matriciel : calculer et vérifier les formes**

In [19]:
wb.record("0B.20a", (A20 @ B20).shape)
wb.record("0B.20b", (B20 @ A20).shape, mistakes={"(m, n) × (n, p) → (m, p) : on garde les dimensions extérieures, ici celles de B puis de A": (3, 3)})
wb.record("0B.20c", B20 @ A20)
wb.record("0B.20d", int((A20 @ B20)[0, 0]))
wb.record("0B.20e", int((A20 @ B20)[2, 1]), mistakes={"(AB)₃₂ : ligne 3 de A, colonne 2 de B": int((A20 @ B20)[1, 2])})
wb.record("0B.20f", C20.shape[1] == A20.shape[0])
wb.record("0B.20g", A20 @ C20)
wb.record("0B.20h", 3 * 2 * 3, mistakes={"9 est le nombre d'éléments de AB ; chacun coûte n multiplications (m × n × p au total)": 9})

WB_ANSWER {"decimals": 0, "element_hashes": ["51a3e32736cfadaf1215afda6241d025a16f14bf8a167192990d454f65abd7d3", "a0a1e7153dcb070ace038dd33432454758c8fb2d6ce35f8b2cc2f87bb5a78914"], "hash": "0f9cbc31c19b3b0745a9725ac82eeddf2cf4dba8da42a52996fb75772cf27cfe", "id": "0B.20a", "integer": true, "kind": "array", "shape": [2]}
📝 Ex 0B.20a : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "element_hashes": ["6f190ec02d431a9a22c677b9f506793b8c987021c29e51aa9227ff9dc493cbb4", "17b95e88c53194f8207552eb299111f044b6ec27415c2f09454fee61120000d0"], "hash": "5cae8e700c675838694dcb86730c19d00a09918123cf306d9a5be2052373c047", "id": "0B.20b", "integer": true, "kind": "array", "mistakes": {"3a51223055141dc143f84f6e1df86e186bfcd8eeea72de4eef813d2f7a5d2551": "(m, n) × (n, p) → (m, p) : on garde les dimensions extérieures, ici celles de B puis de A"}, "shape": [2]}
📝 Ex 0B.20b : réponse enregistrée (array, 0 décimale(s)).
WB_ANSWER {"decimals": 0, "element_hashes": ["8074764d2d9706ce749

**Ex 0B.21 — Identité, inverse 2 × 2 et système de deux équations**

In [20]:
wb.record("0B.21a", np.linalg.det(M21), decimals=0)
wb.record("0B.21b", np.linalg.inv(M21), decimals=1, mistakes={"n'oublie pas de diviser par le déterminant (2)": [[2, -1], [-4, 3]]})
wb.record("0B.21c", np.linalg.solve(M21, [5, 6]), decimals=0)
wb.record("0B.21d", bool(abs(np.linalg.det(np.array([[2, 4], [1, 2]]))) > 1e-12))
wb.record("0B.21e", 6 / 2, decimals=0, mistakes={"det = 1 × 6 − k × 2 = 0": -3})
wb.record("0B.21f", np.linalg.inv(np.array([[2.0, 0], [0, 5]])), decimals=1)

WB_ANSWER {"decimals": 0, "hash": "ad4835edf190df6b1e9e8ae82c9c4d449924e3bc929d47a27f477e9f0b2946dd", "id": "0B.21a", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.21a : réponse enregistrée (float, 0 décimale(s)).
WB_ANSWER {"decimals": 1, "element_hashes": ["56cd36a658f737ec9513289bb2a66347718b8b5dcecec4e0c5454cd729aafb8e", "d5cef14d1b1e4703a982b37f5a7b3e66c8b7c396b13da4d0886e2abc4c759ea2", "3b8b00132e1306e17a1f7106b948857e1be731c23f621e2bee79e05031617d95", "960d79841882505c26e6901dc826ae1c8c3906bd21752c0e07c7427390eb7f5a"], "hash": "612397b1c1b21729120e5c9280cdd4af0c412af339e4ed782741ac5bf5daf32d", "id": "0B.21b", "kind": "array", "mistakes": {"19731e1454ee9a9738a14245ee42ad31d1a9f1a060964f9f0c5eca8d3d812dbb": "n'oublie pas de diviser par le déterminant (2)"}, "shape": [2, 2]}
📝 Ex 0B.21b : réponse enregistrée (array, 1 décimale(s)).
WB_ANSWER {"decimals": 0, "element_hashes": ["3151e3cd3efe684c44ba7b1f37e3b966fc9bb2041b9abebc7394d62da081c54b", "c414637cfbb451631358e0895940abc2

**Ex 0B.22 — Dériver avec les règles : somme, produit, quotient, exp, ln**

In [21]:
wb.record("0B.22a", 12 * 1 ** 2 - 2)
wb.record("0B.22b", 1 / (2 * math.sqrt(4)) - 1 / 4 ** 2, decimals=4, mistakes={"(1/x)' = −1/x² : le signe est négatif": 1 / (2 * math.sqrt(4)) + 1 / 4 ** 2})
wb.record("0B.22c", (2 * 1 + 1 ** 2) * math.e, decimals=3, mistakes={"(uv)' = u'v + uv' : il manque un terme": 2 * math.e})
wb.record("0B.22d", math.log(math.e) + 1, decimals=0)
wb.record("0B.22e", -2 / (3 - 1) ** 2, decimals=1, mistakes={"quotient : (u'v − uv') / v², attention au signe": 0.5})
wb.record("0B.22f", 0)
wb.record("0B.22g", 20 * 2 ** 3, mistakes={"(x⁴)' = 4x³, puis × 5": 80})

WB_ANSWER {"hash": "e6630468098eed155543cdb392b5ea8894b52de9bcd25bfd1e18f23bb2f40db7", "id": "0B.22a", "kind": "int", "magnitude": 1, "sign": 1}
📝 Ex 0B.22a : réponse enregistrée (int).
WB_ANSWER {"decimals": 4, "hash": "0bcb162c05057412ba2afe8aeabc15b4066ef5bb5fc6212452ae57fabb4bc8fb", "hash_coarse": "6edfe3a93f11ba17fe4e76368ecec3f2a6e7721b3c771e90936abb54569c73ca", "id": "0B.22b", "kind": "float", "magnitude": -1, "mistakes": {"6ad66f1dd0922fde505506f38d4bdb1fecbef08e4eea77ded6f92eac6a250015": "(1/x)' = −1/x² : le signe est négatif"}, "sign": 1}
📝 Ex 0B.22b : réponse enregistrée (float, 4 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "02dbbc6ff593e4b4fc912256c4ee625a877c749166486b76386f39b2e1888045", "hash_coarse": "e28a4a25815bed5b6815be4618c478fae358805a3574e4183b9635d6e5ef9007", "id": "0B.22c", "kind": "float", "magnitude": 0, "mistakes": {"95ccde46e40b4172eeff801582fe970cebd808fbceee4ee4a8c2fd2da33b1574": "(uv)' = u'v + uv' : il manque un terme"}, "sign": 1}
📝 Ex 0B.22c : répo

**Ex 0B.23 — Règle de la chaîne : décomposer, puis dériver**

In [22]:
wb.record("0B.23a", 12 * (3 * 1 - 1) ** 3, mistakes={"n'oublie pas la dérivée intérieure (× 3)": 32})
wb.record("0B.23b", 2 * math.exp(1), decimals=3, mistakes={"(e^u)' = e^u × u', avec u' = 2": math.exp(1)})
wb.record("0B.23c", 2 * 2 / (2 ** 2 + 1), decimals=1)
wb.record("0B.23d", 2 / math.sqrt(1 + 4 * 2), decimals=3, mistakes={"(√u)' = u' / (2√u), avec u' = 4": 1 / (2 * math.sqrt(9))})
wb.record("0B.23e", -1 * math.exp(-0.5), decimals=3)
wb.record("0B.23f", -2 * 1 / (1 ** 2 + 1) ** 2, decimals=1)
wb.record("0B.23g", 2 * (3 * 2 + 1 - 4) * 3, mistakes={"dérivée intérieure : d(3w + 1 − 4)/dw = 3": 6})
wb.record("0B.23h", 2 * math.log(math.e) / math.e, decimals=3)

WB_ANSWER {"hash": "17f26fdd4d3eb1a77fe9275ce81f9284bc570569543a6fab0a2d6d9fbf68e440", "id": "0B.23a", "kind": "int", "magnitude": 1, "mistakes": {"7d7cbe692145b62e9d3733f2030b7834e3088bcd8465cb09201907a31de2cdc1": "n'oublie pas la dérivée intérieure (× 3)"}, "sign": 1}
📝 Ex 0B.23a : réponse enregistrée (int).
WB_ANSWER {"decimals": 3, "hash": "e95da47a0c638287e0eaab178b0fb4232a0f3b8a01f4b43c0c5ec1754ca00ff3", "hash_coarse": "42d5703c2633876bfbd43e77762b15efe0fb5e605d623eabd07eb3ead4fd4c71", "id": "0B.23b", "kind": "float", "magnitude": 0, "mistakes": {"a90c67da7f0419d807fc5f58fec66e3867de519e437a87f0aafd13e25b9c265d": "(e^u)' = e^u × u', avec u' = 2"}, "sign": 1}
📝 Ex 0B.23b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "615d69f450fefe4b9d4f63ad0ed6313075899d116d683320e93ca2065969f0a6", "hash_coarse": "63ccd869388bf5567a6c260d03aded170ee74e604486f2be7cde30f29bb79575", "id": "0B.23c", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.23c : réponse e

**Ex 0B.25 — Lignes de niveau, dérivées partielles, gradient et un pas de descente**

In [23]:
wb.record("0B.25a", f25(3, 1))
wb.record("0B.25b", 2 * (3 - 1))
wb.record("0B.25c", 4 * 1, mistakes={"∂(2y²)/∂y = 4y": 2})
wb.record("0B.25d", [2 * (3 - 1), 4 * 1])
wb.record("0B.25e", math.hypot(4, 4), decimals=2)
wb.record("0B.25f", [3 - 0.1 * 4, 1 - 0.1 * 4], decimals=1, mistakes={"on descend : on SOUSTRAIT η∇f": [3.4, 1.4]})
wb.record("0B.25g", f25(3 - 0.1 * 4, 1 - 0.1 * 4), decimals=2)
wb.record("0B.25h", [1, 0])
wb.record("0B.25i", 2 * 1 * 2)
wb.record("0B.25j", 1 ** 2 + 3 * 2 ** 2, mistakes={"∂(x²y)/∂y = x² et ∂(y³)/∂y = 3y²": 12})

WB_ANSWER {"hash": "484d72d1a18d95ae929baa3359b2ed01272997449c0349555c02dba75f9bf54c", "id": "0B.25a", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 0B.25a : réponse enregistrée (int).
WB_ANSWER {"hash": "3a9b2b6c4f1621f044d401baabe6542160740682601f2e81afbe39874c17c7d9", "id": "0B.25b", "kind": "int", "magnitude": 0, "sign": 1}
📝 Ex 0B.25b : réponse enregistrée (int).
WB_ANSWER {"hash": "f76139f3e62c984feee53064ba6289f2b197954d81de231506674a72293ff7fc", "id": "0B.25c", "kind": "int", "magnitude": 0, "mistakes": {"606a244ae30ebeae658a03e26f32934114244cc050e3fed352eb5b04341b4666": "∂(2y²)/∂y = 4y"}, "sign": 1}
📝 Ex 0B.25c : réponse enregistrée (int).
WB_ANSWER {"decimals": 0, "element_hashes": ["26fc0f52fc52e623c9ee1f1b06ada33d9ae5e5608a4c5693d6d08ae30fdee48f", "d72cb64dd3e2350efaf3aca41a5589d481004f4e46d4c5bee1644f2a6f0ee1a8"], "hash": "1d9b283241fd7047d1a1669266dd61efb24ce0c2bd0a863d8f8f4a135c48f575", "id": "0B.25d", "integer": true, "kind": "array", "shape": [2]}
📝 Ex 0B.25d : répons

**Ex 0B.26 — Indépendance : tester P(A ∩ B) = P(A) P(B) avec deux dés**

In [24]:
wb.record("0B.26a", prob(first_even), decimals=3)
wb.record("0B.26b", prob(sum_is(7)), decimals=3)
wb.record("0B.26c", prob(lambda d: first_even(d) and sum_is(7)(d)), decimals=3)
wb.record("0B.26d", math.isclose(prob(lambda d: first_even(d) and sum_is(7)(d)), prob(first_even) * prob(sum_is(7))))
wb.record("0B.26e", prob(sum_is(8)), decimals=3)
wb.record("0B.26f", prob(lambda d: first_even(d) and sum_is(8)(d)), decimals=3)
wb.record("0B.26g", math.isclose(prob(lambda d: first_even(d) and sum_is(8)(d)), prob(first_even) * prob(sum_is(8))))
wb.record("0B.26h", prob(lambda d: first_even(d) and sum_is(8)(d)) - prob(first_even) * prob(sum_is(8)), decimals=3)

WB_ANSWER {"decimals": 3, "hash": "20826e92b4c8d6c922da853cc9d7ef9a4a7be62adf76ffc43b293524c3b21de8", "hash_coarse": "e4d8e9f4edc923ba86c641b1e1c6a5a046663600dfbdf03e8e48138a4c216b49", "id": "0B.26a", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.26a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "f9097c8781d2587eebda321de4dcc5cb52ee4f8a5ad93f683c3b2c5c1e0e513f", "hash_coarse": "fffffda8139e48764a8c1e8b0dacd83cad04841c1b9c45ef329ec316513e7aaf", "id": "0B.26b", "kind": "float", "magnitude": -1, "sign": 1}
📝 Ex 0B.26b : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "38b1dee9e150c725d82401800f67df04c1c674631af5a96fc8077f869719dbdb", "hash_coarse": "0baf2ae08bff0a4e31fcb97c221b639c321a7f95abf99953392ddb8c07ef4e52", "id": "0B.26c", "kind": "float", "magnitude": -2, "sign": 1}
📝 Ex 0B.26c : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"hash": "b5a5bdd36f8e83b1ae22727761b9efc77e5e4ece44c8f560cc6f4a93fd78c785", "

**Ex 0B.27 — Espérance et variance d'une variable discrète**

In [25]:
wb.record("0B.27a", values27 @ probs27, decimals=1)
wb.record("0B.27b", values27 ** 2 @ probs27, decimals=1, mistakes={"E[X²] = Σ x² p (et non E[X]²)": 1.44})
wb.record("0B.27c", values27 ** 2 @ probs27 - (values27 @ probs27) ** 2, decimals=2, mistakes={"Var = E[X²] − E[X]² : n'oublie pas d'élever E[X] au carré": 3.6 - 1.2})
wb.record("0B.27d", math.sqrt(values27 ** 2 @ probs27 - (values27 @ probs27) ** 2), decimals=2)
wb.record("0B.27e", 3 * (values27 @ probs27) + 2, decimals=1)
wb.record("0B.27f", 9 * (values27 ** 2 @ probs27 - (values27 @ probs27) ** 2), decimals=2, mistakes={"Var(aX + b) = a² Var(X) : le b disparaît, le a est au carré": 3 * 2.16 + 2})
wb.record("0B.27g", np.mean([1, 1, 2, 3, 3, 3]), decimals=3)
wb.record("0B.27h", np.var([1, 1, 2, 3, 3, 3]), decimals=3, mistakes={"garde E[X] = 13/6 exact : l'arrondir à 2,167 avant de l'élever au carré fausse le résultat": 5.5 - 2.167 ** 2})

WB_ANSWER {"decimals": 1, "hash": "fe064fe950a33729215794f91e9fede138b8f0798a09694cea5e84af8d864a24", "hash_coarse": "71305ac3ada46eecc9161a0bf5f87c881796ca824b1955dd845f3b756af0adee", "id": "0B.27a", "kind": "float", "magnitude": 0, "sign": 1}
📝 Ex 0B.27a : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 1, "hash": "316e209758f13cfad79affd1e7a1cabcacd54bccece781af4ec860d93d9a38b1", "hash_coarse": "cfaeebe9a7b1a202f19852422b8c8345ae2cca1f3caced9bff08252aecd76b1c", "id": "0B.27b", "kind": "float", "magnitude": 0, "mistakes": {"aeed1e8395d97e3ac91fccc14fa660fc4487739352842e1c5ff865b78dee8331": "E[X²] = Σ x² p (et non E[X]²)"}, "sign": 1}
📝 Ex 0B.27b : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"decimals": 2, "hash": "dcb118a87e3a5f00488d33575d00dd2fb32d0a9e8891edc98339ae4435608315", "hash_coarse": "96c4dbb77eaa17892ede50d9c8a21e04698426564b5724b3cba6111e064d9d60", "id": "0B.27c", "kind": "float", "magnitude": 0, "mistakes": {"c0c9bd1c09e1f82b5e8d01fa6c0eb04

## ✅ Bilan (partie 0)

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu vérifier qu'un produit matriciel existe et donner sa forme, sans calculer ?
2. Sais-tu dériver une composition avec la règle de la chaîne, en nommant les étapes ?
3. Sais-tu calculer une espérance et une variance à partir d'un tableau de probabilités ?

*Les exercices de code 0B.33 à 0B.54 (parties A à D) seront ajoutés à la prochaine session de génération.*